# 05 Generation Examples: PMPGen Results Showcase

Analyze generated peripheral membrane proteins:
- Quality metrics (pLDDT, binding coverage, diversity)
- Sequence and structure characteristics
- Comparison to template proteins
- Success rate by generation stage

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (13, 7)

## 1. Generated Proteins Dataset

In [ ]:
# Synthetic generation results for demonstration
generated_proteins = pd.DataFrame({
    'pdb_id': [f'GEN_{i:04d}' for i in range(100)],
    'sequence_length': np.random.normal(320, 80, 100).astype(int),
    'plddt_mean': np.random.normal(72, 12, 100),
    'plddt_high_conf': np.random.beta(5, 3, 100),
    'binding_coverage': np.random.beta(3, 4, 100),
    'tm_score': np.random.beta(4, 2, 100),
    'rosetta_ddg': np.random.normal(-15, 8, 100),
    'sequence_novelty': np.random.beta(3, 2, 100),
    'passed_plddt_filter': True,
    'passed_binding_filter': True,
    'passed_diversity_filter': True
})

# Add some failures for realism
generated_proteins.loc[np.random.choice(100, 15, replace=False), 'plddt_mean'] = np.random.normal(45, 10, 15)
generated_proteins['passed_plddt_filter'] = generated_proteins['plddt_mean'] > 70

generated_proteins.loc[np.random.choice(100, 12, replace=False), 'binding_coverage'] = np.random.uniform(0.01, 0.2, 12)
generated_proteins['passed_binding_filter'] = (generated_proteins['binding_coverage'] > 0.3) | (generated_proteins['binding_coverage'] < 0.05)

print(f'Total generated proteins: {len(generated_proteins)}')
print(f'\nFirst 10 examples:')
print(generated_proteins.head(10).to_string())

## 2. Generation Quality Summary

In [ ]:
# Quality tiers
high_quality = (generated_proteins['plddt_mean'] > 75) & (generated_proteins['binding_coverage'] > 0.2)
medium_quality = ~high_quality & ((generated_proteins['plddt_mean'] > 70) | (generated_proteins['binding_coverage'] > 0.1))
low_quality = ~high_quality & ~medium_quality

quality_counts = {
    'High Quality': high_quality.sum(),
    'Medium Quality': medium_quality.sum(),
    'Low Quality': low_quality.sum()
}

print('\nGeneration Quality Breakdown:')
for tier, count in quality_counts.items():
    pct = count / len(generated_proteins) * 100
    print(f'{tier:20s}: {count:3d} proteins ({pct:5.1f}%)')

# Pie chart
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Pie chart
colors = ['#4CAF50', '#FFC107', '#F44336']
wedges, texts, autotexts = ax1.pie(quality_counts.values(), labels=quality_counts.keys(),
                                     colors=colors, autopct='%1.1f%%', startangle=90,
                                     textprops={'fontsize': 11, 'fontweight': 'bold'})
ax1.set_title('Generated Protein Quality Distribution', fontsize=13, fontweight='bold')

# Bar chart with counts
ax2.bar(quality_counts.keys(), quality_counts.values(), color=colors, edgecolor='black', linewidth=2)
for i, (k, v) in enumerate(quality_counts.items()):
    ax2.text(i, v + 1, str(v), ha='center', va='bottom', fontweight='bold', fontsize=12)
ax2.set_ylabel('Number of Proteins', fontsize=11, fontweight='bold')
ax2.set_title('Quality Distribution', fontsize=13, fontweight='bold')
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('outputs/generation_quality_tiers.png', dpi=300, bbox_inches='tight')
plt.show()
print('\n✓ Quality tiers plot saved')

## 3. pLDDT Distribution (Foldability)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(generated_proteins['plddt_mean'], bins=20, edgecolor='black', alpha=0.7, color='steelblue')
axes[0].axvline(70, color='green', linestyle='--', linewidth=2, label='Acceptable (70)')
axes[0].axvline(generated_proteins['plddt_mean'].mean(), color='red', linestyle='--', linewidth=2, 
                label=f'Mean: {generated_proteins["plddt_mean"].mean():.1f}')
axes[0].set_xlabel('Mean pLDDT', fontsize=11, fontweight='bold')
axes[0].set_ylabel('Count', fontsize=11, fontweight='bold')
axes[0].set_title('pLDDT Distribution (ESMFold)', fontsize=12, fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3, axis='y')

# Confidence breakdown
conf_levels = {
    'Very High (>90)': (generated_proteins['plddt_mean'] > 90).sum(),
    'High (70-90)': ((generated_proteins['plddt_mean'] > 70) & (generated_proteins['plddt_mean'] <= 90)).sum(),
    'Medium (50-70)': ((generated_proteins['plddt_mean'] > 50) & (generated_proteins['plddt_mean'] <= 70)).sum(),
    'Low (<50)': (generated_proteins['plddt_mean'] <= 50).sum()
}

colors_conf = ['#1B5E20', '#4CAF50', '#FBC02D', '#D32F2F']
axes[1].barh(list(conf_levels.keys()), list(conf_levels.values()), color=colors_conf, edgecolor='black', linewidth=2)
for i, (k, v) in enumerate(conf_levels.items()):
    axes[1].text(v + 1, i, str(v), va='center', fontweight='bold', fontsize=11)
axes[1].set_xlabel('Count', fontsize=11, fontweight='bold')
axes[1].set_title('Confidence Level Breakdown', fontsize=12, fontweight='bold')
axes[1].grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.savefig('outputs/generation_plddt_distribution.png', dpi=300, bbox_inches='tight')
plt.show()
print('✓ pLDDT distribution plot saved')

## 4. Binding Patch Predictions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Binding coverage distribution
axes[0].hist(generated_proteins['binding_coverage'], bins=20, edgecolor='black', alpha=0.7, color='coral')
axes[0].axvline(0.2, color='red', linestyle='--', linewidth=2, label='Target range (0.15-0.35)')
axes[0].axvline(0.35, color='red', linestyle='--', linewidth=2)
axes[0].set_xlabel('Binding Coverage Fraction', fontsize=11, fontweight='bold')
axes[0].set_ylabel('Count', fontsize=11, fontweight='bold')
axes[0].set_title('Binding Patch Coverage Distribution', fontsize=12, fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3, axis='y')

# Binding vs pLDDT scatter
scatter = axes[1].scatter(generated_proteins['plddt_mean'], generated_proteins['binding_coverage'],
                          c=generated_proteins['tm_score'], cmap='viridis', s=100, alpha=0.6,
                          edgecolors='black', linewidth=1)

# Add quality regions
axes[1].axvline(70, color='green', linestyle='--', alpha=0.5, label='pLDDT=70')
axes[1].axhline(0.15, color='red', linestyle='--', alpha=0.5, label='Binding threshold')
axes[1].axhline(0.35, color='red', linestyle='--', alpha=0.5)

axes[1].set_xlabel('Mean pLDDT', fontsize=11, fontweight='bold')
axes[1].set_ylabel('Binding Coverage', fontsize=11, fontweight='bold')
axes[1].set_title('Foldability vs Binding Characteristics', fontsize=12, fontweight='bold')
axes[1].legend(fontsize=10, loc='upper left')
axes[1].grid(True, alpha=0.3)

cbar = plt.colorbar(scatter, ax=axes[1])
cbar.set_label('TM-score', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('outputs/generation_binding_analysis.png', dpi=300, bbox_inches='tight')
plt.show()
print('✓ Binding analysis plot saved')

## 5. Structural Similarity (TM-Score)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# TM-score distribution
axes[0].hist(generated_proteins['tm_score'], bins=15, edgecolor='black', alpha=0.7, color='mediumseagreen')
axes[0].axvline(0.5, color='orange', linestyle='--', linewidth=2, label='Random (0.5)')
axes[0].axvline(generated_proteins['tm_score'].mean(), color='red', linestyle='--', linewidth=2,
                label=f'Mean: {generated_proteins["tm_score"].mean():.3f}')
axes[0].set_xlabel('TM-score', fontsize=11, fontweight='bold')
axes[0].set_ylabel('Count', fontsize=11, fontweight='bold')
axes[0].set_title('TM-score Distribution vs Template', fontsize=12, fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3, axis='y')

# TM-score vs Sequence Novelty
axes[1].scatter(generated_proteins['tm_score'], generated_proteins['sequence_novelty'],
               s=100, alpha=0.6, edgecolors='black', linewidth=1, c='mediumseagreen')
axes[1].set_xlabel('TM-score (Struct. Similarity)', fontsize=11, fontweight='bold')
axes[1].set_ylabel('Sequence Novelty (1 - max identity)', fontsize=11, fontweight='bold')
axes[1].set_title('Structural vs Sequence Divergence', fontsize=12, fontweight='bold')
axes[1].grid(True, alpha=0.3)

# Add trend line
z = np.polyfit(generated_proteins['tm_score'], generated_proteins['sequence_novelty'], 1)
p = np.poly1d(z)
axes[1].plot(generated_proteins['tm_score'].sort_values(), 
            p(generated_proteins['tm_score'].sort_values()), 
            "r--", linewidth=2, label='Trend')
axes[1].legend(fontsize=10)

plt.tight_layout()
plt.savefig('outputs/generation_tm_score.png', dpi=300, bbox_inches='tight')
plt.show()
print('✓ TM-score analysis plot saved')

## 6. Validation Cascade Results

In [ ]:
# Simulate validation cascade
stage1_pass = generated_proteins['passed_plddt_filter'].sum()
stage2_candidates = generated_proteins[generated_proteins['passed_plddt_filter']].copy()
stage2_pass = stage2_candidates['passed_binding_filter'].sum() if len(stage2_candidates) > 0 else 0
stage3_candidates = stage2_candidates[stage2_candidates['passed_binding_filter']].copy()
stage3_pass = stage3_candidates['passed_diversity_filter'].sum() if len(stage3_candidates) > 0 else 0

validation_stages = {
    'Initial\n(N=100)': 100,
    'Stage 1: pLDDT Filter\n(>70)': stage1_pass,
    'Stage 2: DynaMo Binding\nRecall >0.8': stage2_pass,
    'Stage 3: Diversity\n& Rosetta': stage3_pass
}

fig, ax = plt.subplots(figsize=(12, 6))

stages = list(validation_stages.keys())
values = list(validation_stages.values())
percentages = [v / 100 * 100 for v in values]

colors_stages = ['#E0E0E0', '#81C784', '#FFB74D', '#4FC3F7']
bars = ax.bar(stages, values, color=colors_stages, edgecolor='black', linewidth=2, width=0.6)

# Add value labels
for bar, val, pct in zip(bars, values, percentages):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
           f'{int(val)}\n({pct:.0f}%)',
           ha='center', va='bottom', fontweight='bold', fontsize=11)

ax.set_ylabel('Number of Proteins', fontsize=12, fontweight='bold')
ax.set_title('3-Stage Validation Cascade: Generation Success Rate', fontsize=14, fontweight='bold')
ax.set_ylim(0, 110)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('outputs/generation_validation_cascade.png', dpi=300, bbox_inches='tight')
plt.show()
print('✓ Validation cascade plot saved')

## 7. Comprehensive Quality Report

In [ ]:
print('\n' + '='*80)
print('PMPGEN GENERATION QUALITY REPORT')
print('='*80)

print(f'\n[OVERVIEW]')
print(f'{"─"*80}')
print(f'Total generated proteins: {len(generated_proteins)}')
print(f'Generation success rate: {stage3_pass}/{stage1_pass} ({stage3_pass/max(stage1_pass,1)*100:.1f}%)')

print(f'\n[FOLDABILITY]')
print(f'{"─"*80}')
print(f'Mean pLDDT: {generated_proteins["plddt_mean"].mean():.2f} ± {generated_proteins["plddt_mean"].std():.2f}')
print(f'Proteins with pLDDT > 70: {(generated_proteins["plddt_mean"] > 70).sum()} ({(generated_proteins["plddt_mean"] > 70).mean()*100:.1f}%)')
print(f'Proteins with pLDDT > 90: {(generated_proteins["plddt_mean"] > 90).sum()} ({(generated_proteins["plddt_mean"] > 90).mean()*100:.1f}%)')

print(f'\n[BINDING PATCH]')
print(f'{"─"*80}')
print(f'Mean binding coverage: {generated_proteins["binding_coverage"].mean():.2f} ± {generated_proteins["binding_coverage"].std():.2f}')
print(f'Proteins with significant binding patch: {(generated_proteins["binding_coverage"] > 0.15).sum()} ({(generated_proteins["binding_coverage"] > 0.15).mean()*100:.1f}%)')

print(f'\n[STRUCTURAL QUALITY]')
print(f'{"─"*80}')
print(f'Mean TM-score: {generated_proteins["tm_score"].mean():.3f} ± {generated_proteins["tm_score"].std():.3f}')
print(f'TM-score > 0.6 (good similarity): {(generated_proteins["tm_score"] > 0.6).sum()} ({(generated_proteins["tm_score"] > 0.6).mean()*100:.1f}%)')
print(f'Mean Rosetta ΔG: {generated_proteins["rosetta_ddg"].mean():.2f} kcal/mol (favorable binding)')

print(f'\n[SEQUENCE DESIGN]')
print(f'{"─"*80}')
print(f'Mean sequence novelty: {generated_proteins["sequence_novelty"].mean():.2f} (1.0 = completely novel)')
print(f'Completely novel sequences (>90% different): {(generated_proteins["sequence_novelty"] > 0.9).sum()}')

print(f'\n[QUALITY BREAKDOWN]')
print(f'{"─"*80}')
for tier, count in quality_counts.items():
    pct = count / len(generated_proteins) * 100
    print(f'{tier:20s}: {count:3d} proteins ({pct:5.1f}%)')

print(f'\n[RECOMMENDATIONS]')
print(f'{"─"*80}')
if (generated_proteins['plddt_mean'] > 70).mean() > 0.8:
    print('✓ Excellent foldability: Most proteins are predicted to fold well')
else:
    print('⚠ Moderate foldability: Some proteins may need MD optimization')

if (generated_proteins['binding_coverage'] > 0.15).mean() > 0.5:
    print('✓ Good binding patch coverage: Most proteins have predicted binding regions')
else:
    print('⚠ Low binding coverage: Consider adjusting membrane guidance')

if generated_proteins['sequence_novelty'].mean() > 0.7:
    print('✓ High sequence diversity: Generated sequences are novel')
else:
    print('⚠ Low sequence novelty: Results are similar to templates')

print(f'\nNext steps:')
print(f'1. Validate top {max(10, stage3_pass//10)} proteins with MD simulations')
print(f'2. Test binding predictions with wetlab experiments')
print(f'3. Characterize membrane insertion efficiency')
print(f'4. Profile interactions with native membranes')

print('\n' + '='*80)